# 04 · One-stage, anchor-based: RetinaNet

**Model.** torchvision RetinaNet, ResNet-50-FPN v2 (P3–P7), COCO-pretrained; GIoU box loss
(torchvision v2 default), new 8-class sigmoid classification head with focal loss.
**Protocol.** Identical to notebook 03 (fold 0, every 10th frame, 1280 input, 12 epochs, SGD, lr 0.005
for a global batch of 8, 2× T4).

| run | change (one at a time) | motivation |
|---|---|---|
| `retina_base` | torchvision defaults (anchors 32–512 × 3 scales × ratios {0.5, 1, 2}), 500 detections | reference |
| `retina_anchors` | anchors 16–256 (×2^0, 2^⅓, 2^⅔) | covers 13–300 px: the 5th–99th size percentile at 1280 input (EDA §3); ratios kept (person h/w ≈ 2, car ≈ 0.8) |
| `retina_iou` | positive IoU ≥ 0.4, negative < 0.3 (instead of 0.5 / 0.4) | a 12-px object barely reaches IoU 0.5 with any 16-px anchor, so it gets almost no positives |
| `retina_focal` | focal α = 0.5, γ = 1.5 (instead of 0.25 / 2) (loss modification) | fewer, smaller positives: less down-weighting of the hard examples |

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "ivan-in-a-nutshell/AI_HW_2.git"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Data and the fixed benchmark protocol: fold 0, every 10th frame, unmasked test-scale validation frames
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from visdet.data import find_raw_root, find_prepared, read_prepared, fold_split
from visdet.evaluate import build_gt
from visdet.plotting import setup_style, save, SERIES
from visdet.results import load_log, family_table, best_run, plot_histories, plot_breakdown, styled

RAW, PREP = find_raw_root(), find_prepared()
RUNS = Path("/kaggle/working/runs") if ON_KAGGLE else REPO / "runs"
FOLD, STEP = 0, 10
images, boxes, folds = read_prepared(PREP)
train_rows, val_rows = fold_split(images, folds, FOLD, train_step=STEP, val_step=STEP)
gt = build_gt(val_rows, boxes)
N_GPU = torch.cuda.device_count()
SMOKE = os.environ.get("VISDET_SMOKE") == "1"   # CPU dry run: 1 epoch on a few frames, nothing logged for real
if SMOKE:
    val_rows = val_rows.head(8); gt = build_gt(val_rows, boxes)
setup_style()
print(f"fold {FOLD}: {len(train_rows)} training frames, {len(val_rows)} validation frames; {N_GPU} GPU(s)")

In [ ]:
# Anchor count per frame: the imbalance a one-stage detector must handle without a proposal stage
from visdet.config import load_config
from visdet.tv_models import anchor_generator
cfg = load_config("configs/retina_anchors.yaml")
levels = [(int(np.ceil(720 / s)), int(np.ceil(1280 / s))) for s in (8, 16, 32, 64, 128)]
n_anchors = sum(h * w for h, w in levels) * anchor_generator(cfg).num_anchors_per_location()[0]
n_obj = boxes[(boxes.iscrowd == 0) & boxes.image_id.isin(val_rows.image_id)].groupby("image_id").size().median()
print(f"{n_anchors:,} anchors per 1280x720 frame vs {n_obj:.0f} objects (median): ~{n_anchors / n_obj:,.0f} negatives per object")

## Training

In [ ]:
# Train every configuration (skips runs that already have a checkpoint)
from visdet.tv_train import launch
CONFIGS = ['retina_base', 'retina_anchors', 'retina_iou', 'retina_focal']
for name in CONFIGS:
    if not (RUNS / name / "best.pt").exists():
        launch(f"configs/{name}.yaml", PREP, RUNS / name, fold=FOLD, train_step=STEP, val_step=STEP,
               **({"epochs": 1, "limit": 8, "workers": 0} if SMOKE else {}))

## Results and comparison with the tuned two-stage detector

In [ ]:
# RetinaNet runs + the best Faster R-CNN run (from notebook 03, via experiments/results.csv)
log = load_log()
FRCNN = ['frcnn_base', 'frcnn_anchors', 'frcnn_proposals', 'frcnn_giou']
compare = CONFIGS + ([best_run(log, FRCNN)] if any(r in log.index for r in FRCNN) else [])
styled(family_table(log, compare))

In [ ]:
# Learning curves and AP by object size against the tuned Faster R-CNN
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.4))
plot_histories([c for c in CONFIGS if c in log.index], ax=a, title="RetinaNet: validation mAP per epoch")
plot_breakdown(log, [CONFIGS[0], best_run(log, CONFIGS), compare[-1]], ax=b, title="AP by object size")
fig.tight_layout(); save(fig, "retina_results")

## Removing the proposal stage: what can go wrong

1. **Extreme foreground–background imbalance.** RetinaNet classifies *every* anchor (count above),
   whereas Faster R-CNN's RPN reduces them to ≤ 2000 proposals and samples 512 RoIs at 25 % positives.
   Without that filtering, the summed loss is dominated by easy background. *Fix:* focal loss
   (down-weights well-classified anchors, normalised by the number of positives) and a prior-probability
   bias init; the α/γ run tests how much focusing this data needs.
2. **No second look at the box.** Each anchor is classified and regressed once, from features at a fixed
   location; there is no RoIAlign re-pooling and refinement. Localisation of small objects suffers
   (compare AP75 and AP_s with Faster R-CNN). *Fix:* IoU-based regression loss (GIoU), better matched
   anchors, or a refinement stage (cascade / RefineDet-style).
3. **Assignment is all-or-nothing on IoU.** Small objects overlap few anchors above the positive
   threshold, so they get few positives and many become "ignored" (between thresholds). *Fix:* smaller
   anchors and lower thresholds (runs above), or adaptive assignment (ATSS), which is the bridge to the
   anchor-free detectors in notebook 05.
4. **Duplicates.** Many neighbouring anchors fire on one object, so the result depends on NMS. In crowds,
   NMS at IoU 0.5 can also remove true neighbours.

## Failure analysis

Confident detections (score ≥ 0.3) of the best run are labelled with an error type and every object
as found / missed (`visdet.analysis`): background false positives, wrong class, poor localisation,
duplicates, and misses broken down by size and occlusion.

In [ ]:
# Predict the validation frames with the best run and label every error
from visdet.analysis import classify_errors, error_summary, confusion, frame_errors, draw_errors
from visdet.infer import predict_eval_frames, load_frame
from visdet.tv_models import build_model, TorchvisionPredictor
BEST = best_run(log, CONFIGS)
cfg = load_config(f"configs/{BEST}.yaml")
model = build_model(cfg); model.load_state_dict(torch.load(RUNS / BEST / "best.pt", map_location="cpu"))
predictor = TorchvisionPredictor(model, cfg)
dets = predict_eval_frames(predictor, RAW, val_rows, desc="validation")
det_t, gt_t = classify_errors(boxes[boxes.image_id.isin(val_rows.image_id)], dets, score_thr=0.3)
shares, recall = error_summary(det_t, gt_t)
display(shares.to_frame("share of confident detections").T.round(3))
display(recall.round(3))
confusion(det_t)

In [ ]:
# Three failure examples: most missed objects, most class confusions, most background false positives
fe = frame_errors(det_t, gt_t)
picks = {"missed objects": fe.missed.idxmax(),
          "class confusion": fe.get("wrong_cls", fe.missed).idxmax(),
          "background false positives": fe.get("background", fe.missed).idxmax()}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (what, img_id) in zip(axes, picks.items()):
    r = val_rows.set_index("image_id").loc[img_id]
    frame = load_frame(RAW / r.file_name, (r.out_w, r.out_h))
    vis = draw_errors(frame, gt_t[gt_t.image_id == img_id], det_t[det_t.image_id == img_id])
    ax.imshow(vis[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"{what}: {r['clip']} #{r.frame} ({fe.loc[img_id, 'missed']}/{fe.loc[img_id, 'objects']} missed)", fontsize=8)
fig.suptitle("green = found, yellow = missed, red = background FP, orange = wrong class, magenta = poor localisation", fontsize=8)
fig.tight_layout()
save(fig, "retina_failures", photo=True)

## Findings

_Fill in after the runs:_ effect of each tweak (AP_s, AR, AP by density), the initial vs tuned RetinaNet
against the tuned Faster R-CNN (accuracy, small objects, FPS), and whether the issues above show up
in the failure analysis.

**After the session:** download `/kaggle/working/repo/experiments/` (metrics, per-epoch histories) and
`images/` from the notebook output and commit them, so later notebooks and the report can use them.
Checkpoints stay in the notebook output (`/kaggle/working/runs`).